# SIGMOD Exp2 Scan-Mix Crossover

This notebook measures a fixed mixed trace that includes `InitLoad`, `MarkTs`, `Update`, and scan-class transactions.
The trace is deterministic: `20 updates`, `100 scans`, and `readable_every=2`, so at most `10` distinct historical windows are available.
A common untimed history-scan prime runs once immediately after the seed `MarkTs`, and is excluded from the reported latency.
History sweep replaces the first scan after each newly published readable timestamp; delta sweep replaces that same slot with a delta scan over the newly formed adjacent readable pair.


In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
import importlib

ROOT = Path('.').resolve()
sys.path.append(str(ROOT / 'benches'))
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)
from sigmod_exp_common import TOL, apply_paper_style
HTAP_SIM_DIR = ROOT / 'benches/hash_join/htap_simulation'
sys.path.insert(0, str(HTAP_SIM_DIR))
from bench_script_functions import parse_result

apply_paper_style(ROOT)

BIN = ROOT / 'target/release/htap_scanmix_wkld'
OUTDIR = ROOT / 'benches/sigmod_exp2_scanmix_crossover/data'
FIGDIR = ROOT / 'benches/sigmod_exp2_scanmix_crossover/figs'
OUTDIR.mkdir(parents=True, exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'warehouse_count': 7,
    'bucket_num': 4096,
    'update_ratio': 0.0001,
    'readable_every': 2,
    'measured_updates': 20,
    'scans_per_update': 5,
    'repeat': 5,
    'warmup_runs': 1,
    'trim': 1,
    'timeout_sec': 900,
    'history_values': [0.00, 0.01, 0.02, 0.04, 0.06, 0.08, 0.10],
    'delta_values': [0.00, 0.01, 0.02, 0.04, 0.06, 0.08, 0.10],
}

TABLE_ORDER = ['naive', 'ivmh', 'heap', 'chain', 'par']
STYLE = {
    ('naive', ''): ('SNAP', TOL['red'], ':', 'x'),
    ('ivmh', ''): ('IVMH', TOL['yellow'], '--', 'P'),
    ('heap', 'Write Repair'): ('MONO-WR', TOL['blue'], '-', 'o'),
    ('chain', 'Write Repair'): ('DUAL-WR', TOL['cyan'], '-', 's'),
    ('par', 'Write Repair'): ('EPOCH-WR', TOL['green'], '-', 'D'),
}
PLOT_SERIES = [
    ('naive', ''),
    ('ivmh', ''),
    ('heap', 'Write Repair'),
    ('chain', 'Write Repair'),
    ('par', 'Write Repair'),
]

def timed_tx_count(cfg):
    marks_from_updates = cfg['measured_updates'] // max(cfg['readable_every'], 1)
    return 1 + 1 + cfg['measured_updates'] + marks_from_updates + cfg['measured_updates'] * cfg['scans_per_update']

BASE_ARGS = [
    '--warehouse-count', str(CONFIG['warehouse_count']),
    '--bucket-num', str(CONFIG['bucket_num']),
    '--update-ratio', str(CONFIG['update_ratio']),
    '--readable-every', str(CONFIG['readable_every']),
    '--measured-updates', str(CONFIG['measured_updates']),
    '--scans-per-update', str(CONFIG['scans_per_update']),
]

print('ROOT  :', ROOT)
print('BIN   :', BIN)
print('OUTDIR:', OUTDIR)
print('FIGDIR:', FIGDIR)
print('CONFIG:', CONFIG)
print('TIMED_TXS:', timed_tx_count(CONFIG))


In [ ]:
subprocess.run(['cargo', 'build', '--release', '--bin', 'htap_scanmix_wkld'], cwd=ROOT, check=True)
print('Built', BIN)


In [ ]:
def run_checked(args):
    result = subprocess.run(
        [str(x) for x in args],
        cwd=ROOT,
        capture_output=True,
        text=True,
        timeout=CONFIG['timeout_sec'],
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-4000:])
    return result

def aggregate_trial(df):
    if df.empty:
        return pd.DataFrame(columns=['table_type', 'repair_type', 'duration_ms', 'total_ms'])
    out = df.groupby(['table_type', 'repair_type'], dropna=False, as_index=False)['duration_ms'].sum()
    out['total_ms'] = out['duration_ms'] / timed_tx_count(CONFIG)
    return out

def trim_trial_runs(trials):
    if len(trials) <= 2 * CONFIG['trim']:
        return aggregate_trial(pd.concat(trials, ignore_index=True))
    scored = []
    for i, trial in enumerate(trials):
        score = aggregate_trial(trial)['duration_ms'].sum()
        scored.append((score, i))
    scored.sort()
    keep = [i for _, i in scored[CONFIG['trim']:len(scored)-CONFIG['trim']]]
    keep.sort()
    return aggregate_trial(pd.concat([trials[i] for i in keep], ignore_index=True))

def collapse_repairs(df, table_type):
    if table_type in {'naive', 'ivmh'}:
        collapsed = df.groupby(['table_type'], as_index=False)['duration_ms'].mean()
        collapsed['repair_type'] = ''
        collapsed['total_ms'] = collapsed['duration_ms'] / timed_tx_count(CONFIG)
        return collapsed[['table_type', 'repair_type', 'duration_ms', 'total_ms']]
    return df

def run_single_sweep(sweep_type, ratio):
    rows = []
    for table_type in TABLE_ORDER:
        args = [
            str(BIN),
            *BASE_ARGS,
            '--sweep-type', sweep_type,
            '--sweep-ratio', str(ratio),
            '--table-type', table_type,
        ]
        print('Running', sweep_type, ratio, 'table=', table_type)
        for _ in range(CONFIG['warmup_runs']):
            run_checked(args)
        trials = []
        for _trial in range(CONFIG['repeat']):
            result = run_checked(args)
            trials.append(parse_result(result.stdout, table_type))
        df_all = trim_trial_runs(trials)
        df_avg = df_all.groupby(['table_type', 'repair_type'], as_index=False)['duration_ms'].mean()
        df_avg['total_ms'] = df_avg['duration_ms'] / timed_tx_count(CONFIG)
        rows.append(collapse_repairs(df_avg, table_type))
    return pd.concat(rows, ignore_index=True)

def plot_one(ax, df, x_col, xlabel):
    for key in PLOT_SERIES:
        label, color, linestyle, marker = STYLE[key]
        table_type, repair_type = key
        sub = df[(df['table_type'] == table_type) & (df['repair_type'] == repair_type)]
        sub = sub.sort_values(x_col)
        if sub.empty:
            continue
        ax.plot(sub[x_col], sub['total_ms'], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5, label=label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Duration (ms / tx)')
    ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    ax.set_ylim(bottom=0)
    ax.legend(loc='center left', framealpha=0.95)

def render_single(df, x_col, xlabel, stem):
    fig, ax = plt.subplots(1, 1, figsize=(5.0, 4.1))
    plot_one(ax, df, x_col, xlabel)
    fig.tight_layout()
    fig.savefig(FIGDIR / f'{stem}.pdf', format='pdf')
    fig.savefig(FIGDIR / f'{stem}.png', dpi=200)
    plt.show()


In [ ]:
history_frames = []
for value in CONFIG['history_values']:
    df = run_single_sweep('history', value)
    df['history_ratio'] = value
    history_frames.append(df)
df_history = pd.concat(history_frames, ignore_index=True)
df_history['history_pct'] = df_history['history_ratio'] * 100.0
history_csv = OUTDIR / 'scanmix_history.csv'
df_history.to_csv(history_csv, index=False)
display(df_history)
render_single(df_history, 'history_pct', 'Historical Scan Percentage (%)', 'scanmix-history')
print('Saved', history_csv)


In [ ]:
delta_frames = []
for value in CONFIG['delta_values']:
    df = run_single_sweep('delta', value)
    df['delta_ratio'] = value
    delta_frames.append(df)
df_delta = pd.concat(delta_frames, ignore_index=True)
df_delta['delta_pct'] = df_delta['delta_ratio'] * 100.0
delta_csv = OUTDIR / 'scanmix_delta.csv'
df_delta.to_csv(delta_csv, index=False)
display(df_delta)
render_single(df_delta, 'delta_pct', 'Delta Transaction Percentage (%)', 'scanmix-delta')
print('Saved', delta_csv)
